# Reality GPU validation

This notebook is intentionally evidence-first. It fails when a CUDA device or custom kernel launch is unavailable; skipped CUDA tests are not reported as GPU passes.

## A. Verify GPU

Run this before installing Warp so the driver and GPU model are known.

In [ ]:
# ruff: noqa
!nvidia-smi
import platform, sys, subprocess

print("OS:", platform.platform())
print("Python:", sys.version)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout
)

## B. Clone repository

Replace `REPOSITORY_URL` with the repository URL before running this cell.

In [ ]:
# ruff: noqa
REPOSITORY_URL = "https://github.com/REPLACE_ME/reality.git"
REPOSITORY_DIR = "reality"
if "REPLACE_ME" in REPOSITORY_URL:
    raise RuntimeError("Set REPOSITORY_URL to the real GitHub repository before continuing.")
import os, subprocess

if not os.path.isdir(REPOSITORY_DIR):
    subprocess.check_call(["git", "clone", REPOSITORY_URL, REPOSITORY_DIR])
%cd reality

## C. Install Reality dependencies

In [ ]:
# ruff: noqa
%pip install -e '.[dev,physics]'

## D. Install NVIDIA Warp after compatibility inspection

The normal package is attempted first. If it exposes only `cpu`, set `OFFICIAL_WARP_WHEEL_URL` to the CUDA-12-compatible asset URL from the official NVIDIA Warp GitHub release page (`github.com/NVIDIA/warp/releases`) that matches this Python/driver combination. The URL is checked so unofficial binaries are rejected.

In [ ]:
# ruff: noqa
%pip install warp-lang
OFFICIAL_WARP_WHEEL_URL = (
    ""  # paste an official github.com/NVIDIA/warp/releases asset only if needed
)
if OFFICIAL_WARP_WHEEL_URL:
    if not OFFICIAL_WARP_WHEEL_URL.startswith("https://github.com/NVIDIA/warp/releases/download/"):
        raise ValueError("Only official NVIDIA Warp GitHub release wheel URLs are accepted.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", OFFICIAL_WARP_WHEEL_URL])

## E. Verify Warp and select CUDA device

In [ ]:
# ruff: noqa
import os, tempfile

os.environ.setdefault("WARP_CACHE_PATH", os.path.join(tempfile.gettempdir(), "reality-warp-cache"))
import warp as wp

wp.init()
devices = wp.get_devices()
cuda_devices = [device for device in devices if str(device).startswith("cuda")]
print("Warp:", wp.__version__)
print("Devices:", devices)
print("CUDA device count:", len(cuda_devices))
if not cuda_devices:
    raise RuntimeError("Warp exposes only CPU; stop rather than claiming GPU validation.")
selected_device = str(cuda_devices[0])
selected = wp.get_device(selected_device)
print("Selected:", selected_device)
print("Architecture:", getattr(selected, "arch", "unreported"))
print("Compute capability:", getattr(selected, "compute_capability", "unreported"))

## F. Install and test Reality

In [ ]:
# ruff: noqa
!python -m pytest -q

## G. Run deterministic CPU-vs-GPU correctness tests

In [ ]:
# ruff: noqa
!python -m pytest tests/test_gpu.py -q

## H. Run synchronized GPU branch-scale benchmarks

In [ ]:
# ruff: noqa
!python benchmarks/gpu_branch_scaling.py --full

## I. Generate the final GPU audit

The audit writes both Markdown and machine-readable JSON and reports `PARTIAL` if any required GPU proof is unavailable.

In [ ]:
# ruff: noqa
!python tools/reality_gpu_audit.py --full --output REALITY_GPU_AUDIT.md